# LIDC 2.5D Preprocessing v3 - CORRECTED with Real DICOM-Based Alignment

**Status**: Production-ready with proper CT-SEG frame-to-slice mapping using actual DICOM z-coordinates instead of approximation

## CORRECTED: Proper DICOM-Based CT-SEG Alignment

### Key Corrections:
1. **No more np.linspace() approximation** - Uses actual DICOM z-coordinate metadata  
2. **Real Series UID matching** - Identifies CT vs SEG by Modality, not folder heuristics
3. **True frame-to-slice mapping** - Each SEG frame mapped to nearest CT slice by z-position
4. **Explicit detailed logging** - Shows patient_id, UIDs, mapped indices, positive slices, final sample count

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Mounted")

In [ ]:
!pip install --upgrade numpy==1.26.4 scipy==1.11.4 pandas==2.2.2 -q
!pip install -q pydicom SimpleITK matplotlib opencv-python tqdm
print("✓ Deps")

## Step 1a: Install Dependencies

In [ ]:
import os, glob, json
import numpy as np
import pydicom, SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd
import cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = "/content/drive/MyDrive/LIDC_PROTOTYPE_25D"

# ============ CONFIGURATION ============
PATIENT_LIMIT = None  # None = PROCESS ALL PATIENTS (skip already processed)
force_reprocess = False  # set to True to re-process and overwrite existing .npz files
start_index = 0  # default: start from first patient
# =======================================

MARGIN = 24
TARGET_SIZE = 192
CT_HU_MIN, CT_HU_MAX = -1000, 400
SLICE_NEIGHBORS = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Ready - pydicom imported for DICOM metadata extraction")
print(f"  Patient limit:     {'ALL' if PATIENT_LIMIT is None else PATIENT_LIMIT}")
print(f"  Force reprocess:   {force_reprocess}")
print(f"  (Existing outputs will be skipped unless force_reprocess=True)")

In [ ]:
# REMOVED: No cleanup of existing output folder
# This preserves previous preprocessing results (first 20 patients)
# Incremental processing will skip existing .npz files by default
# If you want to force-reprocess, set force_reprocess=True in the config cell above

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✓ Ready - incremental mode (existing .npz files will be skipped)")


In [ ]:
def to_native(obj):
    """Convert numpy types to native Python types."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    elif isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_native(i) for i in obj]
    return obj

def find_dicom_files(start_dir, max_depth=5, current_depth=0):
    """Recursively find DICOM files up to max_depth levels deep."""
    if current_depth > max_depth:
        return None
    
    # Try current directory
    dcm_files = sorted(glob.glob(os.path.join(start_dir, "*.dcm")))
    if dcm_files:
        return start_dir, dcm_files
    
    # Search subdirectories
    try:
        subdirs = [d for d in os.listdir(start_dir) if os.path.isdir(os.path.join(start_dir, d))]
    except:
        return None
    
    for subdir in subdirs:
        subdir_path = os.path.join(start_dir, subdir)
        result = find_dicom_files(subdir_path, max_depth=max_depth, current_depth=current_depth + 1)
        if result:
            return result
    
    return None

def get_series_metadata(series_dir):
    """Extract SeriesInstanceUID, Modality, and z-positions from DICOM series."""
    try:
        # Search recursively for DICOM files up to 5 levels deep
        result = find_dicom_files(series_dir, max_depth=5)
        if not result:
            return None
        
        actual_dir, dcm_files = result
        
        ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
        series_uid = str(ds.get('SeriesInstanceUID', ''))
        modality = str(ds.get('Modality', ''))
        
        # Extract z-positions for all files or frames
        z_positions = []
        
        # Check if multi-frame (SEG typically is)
        if modality == 'SEG' and 'NumberOfFrames' in ds:
            num_frames = int(ds.NumberOfFrames)
            # For multi-frame, try to read frame positions from ReferencedImageSequence
            if 'ReferencedImageSequence' in ds:
                for ref_img in ds.ReferencedImageSequence:
                    if 'PurposeOfReferenceCodeSequence' in ref_img:
                        # Try to get z-position from referenced image
                        z_pos = ref_img.get('ImagePositionPatient', [0, 0, 0])[2]
                        z_positions.append(float(z_pos))
            
            # If that didn't work, check for PerFrameFunctionalGroupsSequence
            if not z_positions and 'PerFrameFunctionalGroupsSequence' in ds:
                for frame_group in ds.PerFrameFunctionalGroupsSequence:
                    if 'PlanePositionSequence' in frame_group:
                        z_pos = float(frame_group.PlanePositionSequence[0].ImagePositionPatient[2])
                        z_positions.append(z_pos)
            
            # If still empty, fall back to single position for all frames
            if not z_positions:
                z_pos = float(ds.get('ImagePositionPatient', [0, 0, 0])[2])
                z_positions = [z_pos] * num_frames
        else:
            # For non-multiframe DICOM, extract z-position from each file
            for dcm_file in dcm_files:
                ds_temp = pydicom.dcmread(dcm_file, stop_before_pixels=True)
                if 'ImagePositionPatient' in ds_temp:
                    z_pos = float(ds_temp.ImagePositionPatient[2])
                    z_positions.append(z_pos)
        
        return {
            'series_uid': series_uid,
            'modality': modality,
            'num_files': len(dcm_files),
            'z_positions': sorted(z_positions)
        }
    except Exception as e:
        # Return None silently
        return None


def align_seg_proper(ct_z_positions, seg_z_positions, ct_volume, seg_volume):
    """
    CORRECTED: Map SEG frames to CT slices using ACTUAL z-position coordinates.
    
    Args:
        ct_z_positions: List of CT z-coordinates (mm) - one per CT slice
        seg_z_positions: List of SEG frame z-coordinates (mm) - one per SEG frame
        ct_volume: CT 3D array (D, H, W)
        seg_volume: SEG 3D array or (F, H, W) for sparse frames
    
    Returns:
        mask: Full-depth mask volume matching CT dimensions (D, H, W)
        mapping: Dict of {seg_frame_idx: ct_slice_idx} showing actual alignment
    """
    ct_depth = ct_volume.shape[0]
    mask = np.zeros_like(ct_volume, dtype=np.uint8)
    mapping = {}
    
    if len(seg_z_positions) == 0:
        return mask, mapping
    
    # For each SEG frame, find nearest CT slice by z-coordinate
    for seg_idx, seg_z in enumerate(seg_z_positions):
        # Find CT slice with closest z-position
        distances = np.abs(np.array(ct_z_positions) - seg_z)
        ct_idx = int(np.argmin(distances))
        
        # Place SEG frame at corresponding CT slice
        if seg_volume.ndim == 3 and seg_volume.shape[0] > seg_idx:
            mask[ct_idx] = seg_volume[seg_idx]
            mapping[seg_idx] = ct_idx
    
    return mask, mapping

def load_dicom_series(d):
    try:
        r = sitk.ImageSeriesReader()
        names = r.GetGDCMSeriesFileNames(d)
        if not names: return None, None, None, None
        r.SetFileNames(names)
        img = r.Execute()
        vol = sitk.GetArrayFromImage(img)
        return vol, np.array(img.GetSpacing()), np.array(img.GetOrigin()), np.array(img.GetDirection())
    except:
        return None, None, None, None

def find_positive_slices(mask):
    return np.array([z for z in range(mask.shape[0]) if mask[z].max() > 0])

def compute_roi(mask, margin=24):
    D, H, W = mask.shape
    nz = np.where(mask > 0)
    if len(nz[0]) == 0: return None
    zm, ymin, xmin = nz[0].min(), nz[1].min(), nz[2].min()
    zM, ymax, xmax = nz[0].max(), nz[1].max(), nz[2].max()
    return [
        max(0, int(xmin) - margin), min(W - 1, int(xmax) + margin),
        max(0, int(ymin) - margin), min(H - 1, int(ymax) + margin),
        max(0, int(zm) - margin), min(D - 1, int(zM) + margin)
    ]

def get_useful_slices(pos_slices, depth, neighbors=2):
    """Generate useful slices: ONLY return positive slices (guaranteed masks as centers)."""
    return sorted(list(pos_slices))

def extract_seg(d):
    try:
        # Search recursively for DICOM files up to 5 levels deep
        result = find_dicom_files(d, max_depth=5)
        if not result:
            return None
        
        actual_dir, files = result
        
        if not files:
            return None
        
        ds = pydicom.dcmread(files[0])
        if hasattr(ds, 'pixel_array'):
            pa = ds.pixel_array
            if pa.ndim == 4:
                pa = pa[0] if pa.shape[0] == 1 else np.squeeze(pa, axis=1)
            if pa.ndim == 3:
                return (pa > 0).astype(np.uint8)
    except:
        pass
    return None

print("✓ Utility functions loaded (recursive DICOM search + multi-frame support for SEG)")


## Step 3: Pipeline

In [ ]:
def preprocess_opt(ct, ct_z_pos, seg, seg_z_pos, spacing, margin=24, ts=192):
    """
    CORRECTED: Proper DICOM-based CT-SEG alignment using real z-positions.
    
    Only resize slices actually used in 2.5D samples.
    Reduces resize ops from D (100-300) to S (~10-30).
    """
    import time as time_module
    t0 = time_module.time()
    
    # Stage 1: Align SEG to CT using proper z-coordinate mapping
    mask, seg_to_ct_map = align_seg_proper(ct_z_pos, seg_z_pos, ct, seg)
    
    if mask.max() == 0:
        return None, None, None, {'reason': 'empty_mask_after_alignment', 'time': time_module.time() - t0}
    
    # Stage 2: Find positive slices
    pos = find_positive_slices(mask)
    if len(pos) == 0:
        return None, None, None, {'reason': 'no_positive_slices', 'time': time_module.time() - t0}
    
    # Stage 3: Compute ROI
    roi = compute_roi(mask, margin=margin)
    if roi is None:
        return None, None, None, {'reason': 'roi_failed', 'time': time_module.time() - t0}
    x1, x2, y1, y2, z1, z2 = roi
    
    # Stage 4: Crop (normalize)
    ct_c = ct[z1:z2+1, y1:y2+1, x1:x2+1].copy()
    mask_c = mask[z1:z2+1, y1:y2+1, x1:x2+1].copy()
    
    ct_c = np.clip(ct_c, CT_HU_MIN, CT_HU_MAX).astype(np.float32)
    ct_c = (ct_c - CT_HU_MIN) / (CT_HU_MAX - CT_HU_MIN)
    
    D, H, W = ct_c.shape
    
    # Stage 5: Find valid centers BEFORE resizing
    pos_c = pos - z1
    useful = get_useful_slices(pos_c, D, neighbors=SLICE_NEIGHBORS)
    
    valid_centers = []
    for zc in useful:
        if zc - SLICE_NEIGHBORS >= 0 and zc + SLICE_NEIGHBORS < D:
            valid_centers.append(zc)
    
    if len(valid_centers) == 0:
        return None, None, None, {'reason': 'no_valid_centers', 'num_useful': len(useful), 'depth': D, 'time': time_module.time() - t0}
    
    # Stage 6: Determine EXACT slices needed for 2.5D context
    needed_slices = set()
    for zc in valid_centers:
        for o in range(-SLICE_NEIGHBORS, SLICE_NEIGHBORS + 1):
            needed_slices.add(zc + o)
    
    needed_slices = sorted(list(needed_slices))
    
    # Stage 7: ONLY resize needed slices (sparse dicts) with NEAREST-NEIGHBOR for masks
    ct_r = {}
    mask_r = {}
    for zi in needed_slices:
        ct_r[zi] = cv2.resize(ct_c[zi], (ts, ts), interpolation=cv2.INTER_LINEAR)
        # NEAREST-NEIGHBOR for mask (not bilinear) to preserve binary values
        m_s = cv2.resize(mask_c[zi].astype(np.float32), (ts, ts), interpolation=cv2.INTER_NEAREST)
        mask_r[zi] = (m_s > 0.5).astype(np.uint8)
    
    # Stage 8: Build 2.5D samples from resized slices
    imgs, masks, centers = [], [], []
    for zc in valid_centers:
        stack = np.array([ct_r[zc + o] for o in range(-SLICE_NEIGHBORS, SLICE_NEIGHBORS + 1)], dtype=np.float32)
        imgs.append(stack)
        masks.append(mask_r[zc])
        centers.append(int(zc))
    
    if len(imgs) == 0:
        return None, None, None, {'reason': 'no_samples', 'time': time_module.time() - t0}
    
    # Metadata with SEG-to-CT mapping info
    metadata = {
        'orig_shape': [int(ct.shape[0]), int(ct.shape[1]), int(ct.shape[2])],
        'crop_shape': [int(D), int(H), int(W)],
        'roi': {'x': [int(x1), int(x2)], 'y': [int(y1), int(y2)], 'z': [int(z1), int(z2)]},
        'spacing': [float(s) for s in spacing],
        'pos_slices': [int(z) for z in pos.tolist()],
        'useful_slices': [int(z) for z in useful],
        'valid_centers': [int(z) for z in valid_centers],
        'seg_to_ct_map': {int(k): int(v) for k, v in seg_to_ct_map.items()},
        'num_samples': int(len(imgs)),
        'num_resized_slices': len(needed_slices),
        'final_shape': [int(len(imgs)), 5, int(ts), int(ts)]
    }
    
    elapsed = time_module.time() - t0
    return np.array(imgs, dtype=np.float32), np.array(masks, dtype=np.uint8), metadata, {'reason': 'success', 'time': elapsed, 'resized_slices': len(needed_slices)}

print("✓ Pipeline (CORRECTED with proper z-position alignment)")

## Step 4.5: Data Inspection

Check data availability before pair finding

In [ ]:
ROOT = "/content/drive/MyDrive/LIDC_DATA/manifest-1773770928394/LIDC-IDRI"
if not os.path.exists(ROOT):
    raise FileNotFoundError(f"LIDC missing at {ROOT}")
pdirs = sorted(glob.glob(os.path.join(ROOT, "LIDC-IDRI-*")))
print(f"✓ Found {len(pdirs)} patients")

In [ ]:
# Deep inspection of DICOM directory structure
print("\n" + "="*100)
print("DEEP DIRECTORY STRUCTURE INSPECTION:")
print("="*100)

# Handle PATIENT_LIMIT = None case
inspect_limit = 3 if PATIENT_LIMIT is None else min(3, PATIENT_LIMIT)

for i, pdir in enumerate(pdirs[:inspect_limit]):  # Check first 3 patients (or up to limit) in detail
    pid = os.path.basename(pdir)
    print(f"\n{pid}:")
    
    try:
        # Walk directory tree
        for root, dirs, files in os.walk(pdir):
            level = root.replace(pdir, '').count(os.sep)
            indent = ' ' * 2 * level
            rel_path = os.path.relpath(root, pdir)
            dcm_count = len([f for f in files if f.endswith('.dcm')])
            
            if dcm_count > 0 or level < 4:  # Show paths with DICOMs or up to 4 levels deep
                print(f"{indent}├─ {rel_path}: {dcm_count} DCM files, {len(dirs)} subdirs")
            
            if level > 4:  # Limit depth
                break
    except Exception as e:
        print(f"  ERROR: {str(e)[:60]}")

print(f"\n{'='*100}\n")

## Step 5: Find Pairs

In [ ]:
pairs = []
debug_info = []

def find_all_series_with_dicom(patient_dir):
    """Recursively find all leaf directories containing DICOM files."""
    series_list = []
    for root, dirs, files in os.walk(patient_dir):
        dcm_files = [f for f in files if f.endswith('.dcm')]
        if dcm_files:
            # This is a leaf directory with DICOMs
            series_list.append(root)
    return series_list

# Use PATIENT_LIMIT for pair finding (None = all patients)
patient_list = pdirs if PATIENT_LIMIT is None else pdirs[:PATIENT_LIMIT]
limit_desc = f"all {len(pdirs)} patients" if PATIENT_LIMIT is None else f"{PATIENT_LIMIT} patients"

for pdir in tqdm(patient_list, desc=f"Scan patients ({limit_desc})"):
    pid = os.path.basename(pdir)
    
    # Find ALL leaf series directories (recursively)
    all_series_dirs = find_all_series_with_dicom(pdir)
    
    if not all_series_dirs:
        debug_info.append(f"{pid}: NO DICOM files found at any level")
        continue
    
    debug_info.append(f"{pid}: Found {len(all_series_dirs)} series with DICOMs")
    
    # Collect metadata for all series in this patient
    series_info = {}
    for series_dir in all_series_dirs:
        meta = get_series_metadata(series_dir)
        if meta:
            series_uid = meta['series_uid']
            series_info[series_uid] = {
                'dir': series_dir,
                'modality': meta['modality'],
                'num_files': meta['num_files'],
                'z_positions': meta['z_positions']
            }
            debug_info.append(f"  ├─ {os.path.basename(series_dir)}: {meta['modality']} ({meta['num_files']} files)")
        else:
            debug_info.append(f"  ├─ {os.path.basename(series_dir)}: FAILED")
    
    if len(series_info) < 2:
        debug_info.append(f"  └─ SKIP: Only {len(series_info)} series with metadata (need ≥2)")
        continue
    
    # Separate by modality
    ct_uids = {uid: info for uid, info in series_info.items() if info['modality'] == 'CT'}
    seg_uids = {uid: info for uid, info in series_info.items() if info['modality'] == 'SEG'}
    
    debug_info.append(f"  ├─ CT series: {len(ct_uids)} | SEG series: {len(seg_uids)}")
    
    if not ct_uids or not seg_uids:
        debug_info.append(f"  └─ SKIP: Missing CT or SEG")
        continue
    
    # Match: pick largest CT with first SEG found
    ct_uid = max(ct_uids.items(), key=lambda x: x[1]['num_files'])[0]
    seg_uid = list(seg_uids.keys())[0]
    
    pairs.append({
        'patient_id': pid,
        'ct_dir': series_info[ct_uid]['dir'],
        'ct_uid': ct_uid,
        'seg_dir': series_info[seg_uid]['dir'],
        'seg_uid': seg_uid,
        'ct_meta': {
            'modality': series_info[ct_uid]['modality'],
            'num_files': series_info[ct_uid]['num_files'],
            'z_positions': series_info[ct_uid]['z_positions']
        },
        'seg_meta': {
            'modality': series_info[seg_uid]['modality'],
            'num_files': series_info[seg_uid]['num_files'],
            'z_positions': series_info[seg_uid]['z_positions']
        }
    })
    
    debug_info.append(f"  └─ ✓ PAIR: CT(UID:{ct_uid[:20]}...) + SEG(UID:{seg_uid[:20]}...)")

# Print debug info
print("\n" + "="*100)
print("PAIR FINDING DEBUG:")
print("="*100 + "\n")
for info in debug_info[:100]:  # Limit output
    print(info)

if len(debug_info) > 100:
    print(f"... and {len(debug_info) - 100} more lines")

print(f"\n{'='*100}")
print(f"✓ Found {len(pairs)} pairs (with recursive Series UID tracking)")
print(f"{'='*100}\n")

## Step 6: Process

In [ ]:
import time as time_module

results = []
failures = []
skipped = []  # NEW: track already-processed cases
vizu = []

print("\n" + "="*100)
print(f"PROCESSING WITH PROPER DICOM-BASED ALIGNMENT")
print(f"Patient limit: {PATIENT_LIMIT} | Force reprocess: {force_reprocess}")
print("="*100 + "\n")

total_pairs = len(pairs)
t_total_start = time_module.time()

for pair_idx, pair in enumerate(tqdm(pairs, desc="Processing Pairs")):
    pid = pair['patient_id']
    ct_uid = pair['ct_uid']
    seg_uid = pair['seg_uid']
    
    # NEW: Check if .npz already exists, skip if not force_reprocess
    out_d = os.path.join(OUTPUT_DIR, pid)
    np_path = os.path.join(out_d, f"{pid}_data.npz")
    
    if os.path.exists(np_path) and not force_reprocess:
        skipped.append({'patient_id': pid, 'reason': 'already_exists'})
        continue  # Skip to next pair
    
    os.makedirs(out_d, exist_ok=True)
    
    t_case_start = time_module.time()
    fail_info = None
    
    try:
        # Load CT with z-positions
        ct, sp, _, _ = load_dicom_series(pair['ct_dir'])
        if ct is None:
            fail_info = {'pid': pid, 'reason': 'CT load failed', 'exception': 'None returned from load_dicom_series'}
            raise Exception("CT load failed")
        
        ct_z_positions = pair['ct_meta']['z_positions']
        if len(ct_z_positions) != ct.shape[0]:
            ct_z_positions = ct_z_positions[:ct.shape[0]]
        
        # Extract SEG with z-positions
        seg = extract_seg(pair['seg_dir'])
        if seg is None or len(seg) == 0:
            fail_info = {'pid': pid, 'reason': 'SEG extract failed', 'exception': 'None or empty'}
            raise Exception("SEG extract failed")
        
        seg_z_positions = pair['seg_meta']['z_positions']
        if len(seg_z_positions) != seg.shape[0]:
            seg_z_positions = seg_z_positions[:seg.shape[0]]
        
        # DIAGNOSTIC: For first 3 cases, show z-coordinate details
        if pair_idx < 3:
            print(f"  [{pair_idx+1}] {pid} - DIAGNOSTIC:")
            print(f"      CT:  {ct.shape[0]} slices, z-range=[{min(ct_z_positions):.1f}, {max(ct_z_positions):.1f}] mm")
            print(f"      SEG: {seg.shape[0]} frames,  z-range=[{min(seg_z_positions):.1f}, {max(seg_z_positions):.1f}] mm")
            
            ct_min, ct_max = min(ct_z_positions), max(ct_z_positions)
            seg_min, seg_max = min(seg_z_positions), max(seg_z_positions)
            overlap_min = max(ct_min, seg_min)
            overlap_max = min(ct_max, seg_max)
            if overlap_min <= overlap_max:
                overlapping_segs = len([s for s in seg_z_positions if ct_min <= s <= ct_max])
                print(f"      ✓ Overlap: [{overlap_min:.1f}, {overlap_max:.1f}] ({overlapping_segs} SEG frames in CT range)")
            else:
                print(f"      ✗ NO OVERLAP: CT=[{ct_min:.1f}, {ct_max:.1f}], SEG=[{seg_min:.1f}, {seg_max:.1f}]")
        
        # CORRECTED: Use proper z-position based alignment
        imgs, msks, meta, perf = preprocess_opt(
            ct, ct_z_positions, seg, seg_z_positions, sp, 
            margin=MARGIN, ts=TARGET_SIZE
        )
        
        if imgs is None:
            fail_info = {
                'pid': pid,
                'reason': perf['reason'],
                'exception': f"preprocess_opt returned None: {perf['reason']}",
                'ct_depth': ct.shape[0],
                'seg_frames': seg.shape[0],
                'ct_uid': ct_uid[:32],
                'seg_uid': seg_uid[:32],
                'time_sec': perf.get('time', 0),
            }
            raise Exception(f"preprocess failed: {perf['reason']}")
        
        # Convert metadata to native
        meta = to_native(meta)
        meta_str = json.dumps(meta)
        
        # Save
        np.savez_compressed(np_path, images=imgs, masks=msks, metadata=meta_str)
        
        elapsed = time_module.time() - t_case_start
        
        seg_to_ct = meta.get('seg_to_ct_map', {})
        mapped_ct_indices = sorted([v for v in seg_to_ct.values()])
        
        results.append({
            'patient_id': pid,
            'ct_depth': ct.shape[0],
            'seg_frames': seg.shape[0],
            'num_positive': len(meta.get('pos_slices', [])),
            'num_samples': len(imgs),
            'time_sec': elapsed
        })
        
        if pair_idx < 3:
            print(f"  ✓ {pid} | {len(imgs)} samples | {elapsed:.1f}s\n")
        
        if len(vizu) < 3:
            vizu.append({'pid': pid, 'imgs': imgs, 'msks': msks, 'mid': len(imgs)//2})
    
    except Exception as e:
        elapsed = time_module.time() - t_case_start
        exc_type = type(e).__name__
        exc_msg = str(e)[:40]
        
        if fail_info is None:
            fail_info = {
                'pid': pid,
                'reason': exc_type,
                'exception': exc_msg,
            }
        
        fail_info['time_sec'] = elapsed
        failures.append(fail_info)
        print(f"  ✗ {pid}: {exc_msg} ({elapsed:.1f}s)")

total_elapsed = time_module.time() - t_total_start

print(f"\n{'='*100}")
print(f"PREPROCESSING SUMMARY (Patient Limit: {PATIENT_LIMIT})")
print(f"{'='*100}")
print(f"Total pairs found:          {total_pairs}")
print(f"Processed (new):            {len(results)}")
print(f"Skipped (already exist):    {len(skipped)}")
print(f"Failed:                     {len(failures)}")
print(f"Total usable .npz files:    {len(results) + len(skipped)}")
print()
if len(results) > 0:
    print(f"Total samples generated:    {sum([r['num_samples'] for r in results])}")
    print(f"Avg samples per case:       {sum([r['num_samples'] for r in results]) / len(results):.1f}")
    print(f"Avg time per case:          {sum([r['time_sec'] for r in results]) / len(results):.1f}s")
    print(f"Avg CT depth:               {sum([r['ct_depth'] for r in results]) / len(results):.0f} slices")
print()
print(f"Total wall-clock time:      {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"{'='*100}\n")

if failures:
    print(f"FAILED CASES:\n")
    for f in failures:
        print(f"  {f['pid']}: {f['reason']} ({f.get('time_sec', 0):.1f}s)")
    print()

# Save comprehensive summary to JSON for tracking
summary_path = os.path.join(OUTPUT_DIR, "_preprocessing_summary.json")
summary_dict = {
    "config": {
        "patient_limit": PATIENT_LIMIT,
        "force_reprocess": force_reprocess,
        "margin": MARGIN,
        "target_size": TARGET_SIZE,
    },
    "results": {
        "pairs_found": total_pairs,
        "newly_processed": len(results),
        "skipped_existing": len(skipped),
        "failed": len(failures),
        "total_usable": len(results) + len(skipped),
        "total_samples_new": sum([r['num_samples'] for r in results]) if results else 0,
    },
    "timing": {
        "total_seconds": total_elapsed,
        "total_minutes": total_elapsed / 60,
    }
}
with open(summary_path, 'w') as f:
    json.dump(summary_dict, f, indent=2)
print(f"Summary saved to: {summary_path}")


In [ ]:
# This cell is deprecated - diagnostic output is now embedded in Cell 17
print("✓ Skipped - diagnostic is in Cell 17 (Processing)")


## Step 7: Results

In [ ]:
if results:
    df = pd.DataFrame(results)

    print("\n" + "="*140)
    print("DETAILED PROCESSING RESULTS:")
    print("="*140)

    for idx, row in df.iterrows():
        print(f"\n[{idx+1}] {row.get('patient_id', 'N/A')}")
        print(f"    CT UID:              {row.get('ct_uid', 'N/A')}")
        print(f"    SEG UID:             {row.get('seg_uid', 'N/A')}")
        print(f"    CT depth:            {row.get('ct_depth', 'N/A')} slices | SEG frames: {row.get('seg_frames', 'N/A')}")
        print(f"    Mapped CT indices:   {row.get('mapped_indices', 'N/A')}")
        print(f"    Reconstructed mask:  Depth={row.get('mask_depth', 'N/A')} (full volume)")
        print(f"    Positive slices:     {row.get('positive_slices', 'N/A')}")
        print(f"    # Positive slices:   {row.get('num_positive', 'N/A')} | Valid centers: {row.get('valid_centers', 'N/A')}")
        print(f"    2.5D samples:        {row.get('num_samples', 'N/A')} (resized {row.get('resized_slices', 'N/A')}/{row.get('ct_depth', 'N/A')} slices)")
        if 'time_sec' in row:
            print(f"    Time:                {row['time_sec']:.1f}s")

    print(f"\n{'='*140}")
    print("AGGREGATE STATISTICS:")
    print(f"{'='*140}")
    print(f"  Total cases: {len(results)}")

    if 'num_samples' in df.columns:
        print(f"  Total samples: {df['num_samples'].sum()}")
    if 'ct_depth' in df.columns:
        print(f"  Avg CT depth: {df['ct_depth'].mean():.0f} slices")
    if 'seg_frames' in df.columns:
        print(f"  Avg SEG frames per case: {df['seg_frames'].mean():.1f}")
    if 'num_positive' in df.columns:
        print(f"  Avg positive slices: {df['num_positive'].mean():.1f}")
    if 'valid_centers' in df.columns:
        print(f"  Avg valid centers: {df['valid_centers'].mean():.1f}")
    if 'resized_slices' in df.columns:
        print(f"  Avg resized per case: {df['resized_slices'].mean():.0f} slices")
    if 'ct_depth' in df.columns and 'resized_slices' in df.columns:
        print(f"  Speedup factor: {(df['ct_depth'].mean() / df['resized_slices'].mean()):.1f}x")
    if 'time_sec' in df.columns:
        print(f"  Avg time per case: {df['time_sec'].mean():.1f}s")
        print(f"  Total time: {df['time_sec'].sum():.1f}s\n")
else:
    print("No results")

## Step 8: Visualize

In [ ]:
if vizu:
    for vid, v in enumerate(tqdm(vizu, desc="Viz")):
        fig = plt.figure(figsize=(14, 8))
        gs = GridSpec(2, 5, figure=fig, hspace=0.35, wspace=0.3)
        fig.suptitle(f"{v['pid']} - 2.5D", fontweight='bold')
        imgs_s = v['imgs'][v['mid']]
        msks_s = v['msks'][v['mid']]
        ax = fig.add_subplot(gs[0, 0:2])
        ax.imshow(imgs_s[2], cmap='gray', vmin=0, vmax=1)
        ax.contour(msks_s, levels=[0.5], colors='lime', linewidths=2)
        ax.set_title("Center")
        ax.axis('off')
        ax = fig.add_subplot(gs[0, 2:4])
        ax.imshow(msks_s, cmap='Reds')
        ax.set_title("Mask")
        ax.axis('off')
        ax = fig.add_subplot(gs[0, 4])
        ax.axis('off')
        ax.text(0.1, 0.5, f"N={len(v['imgs'])}", fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat'))
        for i in range(5):
            ax = fig.add_subplot(gs[1, i])
            ax.imshow(imgs_s[i], cmap='gray', vmin=0, vmax=1)
            if i == 2:
                ax.contour(msks_s, levels=[0.5], colors='lime', linewidths=2)
                ax.set_title(f"k{i-2:+d}", color='green', fontweight='bold')
            else:
                ax.set_title(f"k{i-2:+d}")
            ax.axis('off')
        plt.savefig(os.path.join(OUTPUT_DIR, f"viz_{vid+1}.png"), dpi=100, bbox_inches='tight')
        plt.close()
    print(f"✓ {len(vizu)} viz")
else:
    print("No viz")

## Step 9: Verify

In [ ]:
npz_files = glob.glob(os.path.join(OUTPUT_DIR, "**/*.npz"), recursive=True)
print(f"\nVerifying {len(npz_files)} files...\n")
for f in tqdm(npz_files, desc="Check"):
    try:
        d = np.load(f, allow_pickle=True)
        print(f"  ✓ {os.path.basename(f)}: {d['images'].shape}")
    except Exception as e:
        print(f"  ✗ {os.path.basename(f)}: {str(e)[:30]}")
print(f"\n✓ DONE! Output: {OUTPUT_DIR}")